In [1]:
# Cell 1: Environment Setup
!pip install pandas numpy scikit-learn matplotlib pyarrow -q

import pandas as pd
import numpy as np
import os, json, csv, time, random, logging
from datetime import date, datetime, timedelta

random.seed(42)
np.random.seed(42)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
print("✓ Environment ready for: CTEs and Window Functions")

✓ Environment ready for: CTEs and Window Functions


In [2]:
# Cell 2: Generate Dataset
PRODUCTS = ['Laptop','Monitor','Keyboard','Mouse','Headset','Webcam','USB Hub','Desk Lamp','Speaker','Tablet']
REGIONS = ['North','South','East','West']
STATUSES = ['completed','pending','cancelled','refunded']

records = []
for i in range(10000):
    records.append({
        'order_id': f'ORD-{i:05d}',
        'product': random.choice(PRODUCTS),
        'category': PRODUCTS[i % len(PRODUCTS)].split()[0] if i < 50 else random.choice(['Electronics','Accessories','Home','Sports','Office']),
        'region': random.choice(REGIONS),
        'quantity': random.randint(1, 50),
        'unit_price': round(random.uniform(9.99, 999.99), 2),
        'order_date': str(date(2024,1,1) + timedelta(days=random.randint(0,364))),
        'status': random.choices(STATUSES, weights=[70,15,10,5])[0]
    })

df = pd.DataFrame(records)
df['revenue'] = (df['quantity'] * df['unit_price']).round(2)

print(f"✓ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

✓ Dataset loaded: 10000 rows, 9 columns


,order_id,product,category,region,quantity,unit_price,order_date,status,revenue
0,ORD-00000,Monitor,Laptop,North,48,282.27,2024-04-24,completed,13548.96
1,ORD-00001,Monitor,Monitor,North,38,427.69,2024-01-16,completed,16252.22
2,ORD-00002,Mouse,Keyboard,North,36,206.84,2024-11-28,pending,7446.24
3,ORD-00003,USB Hub,Mouse,South,29,593.36,2024-01-04,pending,17207.44
4,ORD-00004,Keyboard,Headset,West,22,285.08,2024-04-20,refunded,6271.76


In [3]:
# Cell 3: Data Profiling
print('=' * 60)
print('DATA PROFILE: CTEs and Window Functions')
print('=' * 60)
print(f'\nShape: {df.shape}')
print(f'\nColumn Types:\n{df.dtypes}')
print(f'\nNull Counts:\n{df.isnull().sum()}')
print(f'\nDuplicate Rows: {df.duplicated().sum()}')
print(f'\nNumeric Summary:\n{df.describe()}')
print(f'\nUnique Values:')
for col in df.columns:
    print(f'  {col}: {df[col].nunique()} unique')
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

DATA PROFILE: CTEs and Window Functions

Shape: (10000, 9)

Column Types:
order_id       object
product        object
category       object
region         object
quantity        int64
unit_price    float64
order_date     object
status         object
revenue       float64
dtype: object

Null Counts:
order_id      0
product       0
category      0
region        0
quantity      0
unit_price    0
order_date    0
status        0
revenue       0
dtype: int64

Duplicate Rows: 0

Numeric Summary:
           quantity    unit_price       revenue
count  10000.000000  10000.000000  10000.000000
mean      25.140600    503.807532  12629.256019
std       14.580209    287.654131  11033.664281
min        1.000000     10.050000     16.440000
25%       12.000000    254.082500   3529.290000
50%       25.000000    502.315000   9347.180000
75%       38.000000    751.495000  19358.000000
max       50.000000    999.970000  49685.500000

Unique Values:
  order_id: 10000 unique
  product: 10 unique
  category: 

In [4]:
# Cell 4: Core Implementation — CTEs and Window Functions

import sqlite3

# Load data into SQLite for SQL-based CTE and window function demos
conn = sqlite3.connect(':memory:')
df.to_sql('orders', conn, index=False, if_exists='replace')

# === CTE (Common Table Expression) using WITH clause ===
# CTEs make complex queries readable by breaking them into named steps

# CTE 1: Monthly revenue summary
print('=== 1. CTE — Monthly revenue by product ===')
result = pd.read_sql('''
    WITH monthly_product_revenue AS (
        SELECT product,
               SUBSTR(order_date, 1, 7) AS month,
               SUM(revenue) AS monthly_rev,
               COUNT(*) AS order_count
        FROM orders
        WHERE status = 'completed'
        GROUP BY product, SUBSTR(order_date, 1, 7)
    )
    SELECT product, month, ROUND(monthly_rev, 2) AS revenue, order_count
    FROM monthly_product_revenue
    ORDER BY monthly_rev DESC
    LIMIT 10
''', conn)
print(result)

# CTE 2: Chained CTEs — average then filter
print('\n=== 2. Chained CTEs — Products above average revenue ===')
result = pd.read_sql('''
    WITH product_totals AS (
        SELECT product, SUM(revenue) AS total_rev
        FROM orders WHERE status = 'completed'
        GROUP BY product
    ),
    avg_rev AS (
        SELECT AVG(total_rev) AS overall_avg FROM product_totals
    )
    SELECT pt.product, ROUND(pt.total_rev, 2) AS total,
           ROUND(ar.overall_avg, 2) AS avg_all,
           ROUND(pt.total_rev - ar.overall_avg, 2) AS diff
    FROM product_totals pt, avg_rev ar
    WHERE pt.total_rev > ar.overall_avg
    ORDER BY pt.total_rev DESC
''', conn)
print(result)

# === Window Functions ===
# Window functions compute values across rows related to the current row

# RANK: rank products by revenue within each region
print('\n=== 3. RANK() — Top product per region ===')
result = pd.read_sql('''
    SELECT region, product, ROUND(total_rev, 2) AS revenue, rnk
    FROM (
        SELECT region, product, SUM(revenue) AS total_rev,
               RANK() OVER (PARTITION BY region ORDER BY SUM(revenue) DESC) AS rnk
        FROM orders WHERE status = 'completed'
        GROUP BY region, product
    )
    WHERE rnk <= 3
    ORDER BY region, rnk
''', conn)
print(result)

# ROW_NUMBER: assign row numbers within groups
print('\n=== 4. ROW_NUMBER() — Sequential numbering per region ===')
result = pd.read_sql('''
    SELECT region, order_id, revenue,
           ROW_NUMBER() OVER (PARTITION BY region ORDER BY revenue DESC) AS row_num
    FROM orders
    WHERE status = 'completed'
    LIMIT 12
''', conn)
print(result)

# Running total (cumulative SUM)
print('\n=== 5. SUM() OVER — Running total by date ===')
result = pd.read_sql('''
    WITH daily AS (
        SELECT order_date, SUM(revenue) AS daily_rev
        FROM orders WHERE status = 'completed'
        GROUP BY order_date
    )
    SELECT order_date, ROUND(daily_rev, 2) AS daily_revenue,
           ROUND(SUM(daily_rev) OVER (ORDER BY order_date), 2) AS running_total
    FROM daily
    ORDER BY order_date
    LIMIT 15
''', conn)
print(result)

# LAG/LEAD: compare with previous/next period
print('\n=== 6. LAG() — Month-over-month revenue change ===')
result = pd.read_sql('''
    WITH monthly AS (
        SELECT SUBSTR(order_date, 1, 7) AS month, SUM(revenue) AS rev
        FROM orders WHERE status = 'completed'
        GROUP BY SUBSTR(order_date, 1, 7)
    )
    SELECT month,
           ROUND(rev, 2) AS revenue,
           ROUND(LAG(rev) OVER (ORDER BY month), 2) AS prev_month,
           ROUND(rev - LAG(rev) OVER (ORDER BY month), 2) AS change
    FROM monthly
    ORDER BY month
''', conn)
print(result)

# NTILE: divide into quartiles
print('\n=== 7. NTILE(4) — Revenue quartiles ===')
result = pd.read_sql('''
    SELECT order_id, product, revenue,
           NTILE(4) OVER (ORDER BY revenue) AS quartile
    FROM orders
    WHERE status = 'completed'
    ORDER BY revenue DESC
    LIMIT 10
''', conn)
print(result)

conn.close()
print('\n-- CTEs and Window Functions implementation complete')

=== 1. CTE — Monthly revenue by product ===
    product    month     revenue  order_count
0    Tablet  2024-07  1153418.90           82
1   USB Hub  2024-03  1035509.33           88
2   USB Hub  2024-06  1004351.93           72
3    Tablet  2024-02   999303.08           72
4  Keyboard  2024-07   966785.14           71
5    Tablet  2024-09   957165.12           68
6   Monitor  2024-10   950400.03           68
7    Webcam  2024-10   944750.01           60
8   USB Hub  2024-01   936285.70           68
9  Keyboard  2024-11   918239.71           70

=== 2. Chained CTEs — Products above average revenue ===
    product       total     avg_all       diff
0    Tablet  9657785.16  8810651.75  847133.41
1   USB Hub  9492763.41  8810651.75  682111.66
2    Webcam  9112511.08  8810651.75  301859.33
3   Speaker  8996319.79  8810651.75  185668.04
4  Keyboard  8902874.72  8810651.75   92222.97
5   Monitor  8892348.28  8810651.75   81696.53

=== 3. RANK() — Top product per region ===
   region    produc

In [5]:
# Cell 5: Results Analysis
print('=' * 60)
print('RESULTS: CTEs and Window Functions')
print('=' * 60)
print(f'Input rows: {len(df)}')
print(f'Processing complete')

# Display key metrics
print(f'\nRevenue by Region:')
print(df.groupby('region')['revenue'].sum().sort_values(ascending=False))

RESULTS: CTEs and Window Functions
Input rows: 10000
Processing complete

Revenue by Region:
region
West     32704052.11
East     31929463.06
South    30902956.65
North    30756088.37
Name: revenue, dtype: float64


In [6]:
# Cell 6: Self-Check — CTEs and Window Functions
# Run this cell to verify your work before clicking "Run Tests"
print('=' * 50)
print('SELF-CHECK — CTEs and Window Functions')
print('=' * 50)

checks = {
    'DataFrame exists and is not empty': len(df) > 0,
    'Has at least 2 columns': len(df.columns) >= 2,
    'No fully-null columns': df.isnull().mean().max() < 0.5,
    'Has at least 10 rows': len(df) >= 10,
    'Data types look valid': df.dtypes is not None,
}

passed = sum(1 for v in checks.values() if v)
for name, ok in checks.items():
    print(f'  {"PASS" if ok else "FAIL"}: {name}')

print(f'\n{passed}/{len(checks)} self-checks passed')
if passed == len(checks):
    print('\nAll good! Click "Run Tests" to submit for official validation.')
else:
    print('\nSome checks failed. Fix your code, then click "Run Tests".')

SELF-CHECK — CTEs and Window Functions
  PASS: DataFrame exists and is not empty
  PASS: Has at least 2 columns
  PASS: No fully-null columns
  PASS: Has at least 10 rows
  PASS: Data types look valid

5/5 self-checks passed

All good! Click "Run Tests" to submit for official validation.
